# Préparation des données - Togo MICS6

Notebook de lecture, sélection des variables et nettoyage préliminaire des
fichiers `ch.sav`, `hh.sav` et `wm.sav` (Togo MICS6, INSEED-UNICEF).

**Ce que fait ce notebook :**
1. Lecture des trois fichiers d'origine et affichage de leurs dimensions
2. Sélection des variables retenues, par fichier, avec explication
3. Décodage des principales variables catégorielles
4. Recodage des valeurs manquantes (7/8/9)
5. Jointure des trois fichiers en un seul jeu de données, avec contrôle
6. Sauvegarde du résultat

**Ce que ce notebook ne fait pas** : la construction de la variable cible et la
liste des variables à exclure pour éviter la fuite de données (data leakage)
font l'objet d'une étape et d'une note séparées.


## 1. Chemins et lecture des fichiers d'origine

Structure de dossiers attendue (ce notebook est placé au même niveau que `Stunting_Data/`) :

```
Stunting/
├── Stunting_Data/
│   ├── Datasets_originaux/     <- ch.sav, hh.sav, wm.sav
│   └── Datasets_convertis/     <- CSV et Parquet générés par ce notebook
└── Stunting_Notebook/
│   ├── preparation_donnees.ipynb/ 
```


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Dossiers, relatifs à l'emplacement de ce notebook
DOSSIER_DATA = Path("../Stunting_Data")
DOSSIER_ORIGINAUX = DOSSIER_DATA / "Datasets_originaux"
DOSSIER_CONVERTIS = DOSSIER_DATA / "Datasets_convertis"

DOSSIER_CONVERTIS.mkdir(parents=True, exist_ok=True)

# convert_categoricals=False : on garde les codes numériques bruts (1, 2, 9...)
# plutôt que les libellés texte, pour rester cohérent avec le décodage manuel
# fait plus bas dans ce notebook.

ch = pd.read_spss(DOSSIER_ORIGINAUX / "ch.sav", convert_categoricals=False)
hh = pd.read_spss(DOSSIER_ORIGINAUX / "hh.sav", convert_categoricals=False)
wm = pd.read_spss(DOSSIER_ORIGINAUX / "wm.sav", convert_categoricals=False)

print("Dimensions des fichiers d'origine")
print("-" * 40)
print(f"ch.sav (enfants)  : {ch.shape[0]} lignes, {ch.shape[1]} colonnes")
print(f"hh.sav (ménages)  : {hh.shape[0]} lignes, {hh.shape[1]} colonnes")
print(f"wm.sav (mères)    : {wm.shape[0]} lignes, {wm.shape[1]} colonnes")


Dimensions des fichiers d'origine
----------------------------------------
ch.sav (enfants)  : 5030 lignes, 467 colonnes
hh.sav (ménages)  : 8404 lignes, 297 colonnes
wm.sav (mères)    : 7657 lignes, 423 colonnes


## 2. Variables retenues, par fichier

### `hh.sav` - Ménage

| Code | Signification |
|------|----------------|
| `HH1`, `HH2` | Identifiants (grappe, ménage) - nécessaires pour la jointure |
| `HH6` | Milieu de résidence |
| `HH7` | Région administrative |
| `WS1` | Source principale d'eau de boisson |
| `WS11` | Type de toilettes / installation sanitaire |
| `windex5` | Quintile de bien-être du ménage (indice de richesse) |
| `HH48` | Nombre de membres du ménage |

### `ch.sav` - Enfant de moins de 5 ans (table pivot)

| Code | Signification |
|------|----------------|
| `HH1`, `HH2`, `AN3`, `AN5` | Identifiants (grappe, ménage, enfant, mère) - pas des questions, juste pour la jointure |
| `HL4` | Sexe de l'enfant |
| `CAGE` | Âge de l'enfant, en mois |
| `melevel` | Niveau d'instruction de la mère |
| `BD3` | L'enfant est encore allaité |
| `CA1` | Épisode de diarrhée dans les deux dernières semaines |
| `CA14` | Épisode de fièvre dans les deux dernières semaines |
| `HAZ2`, `WAZ2`, `WHZ2` | Indices anthropométriques OMS 2006 - pas des questions non plus : utilisés uniquement pour construire la cible à l'entraînement, jamais mesurés sur le terrain à l'usage |
| `HAZFLAG`, `WAZFLAG`, `WHZFLAG`, `FLAG` | Indicateurs de qualité de la mesure |

**`BD2`** (l'enfant a déjà été allaité) a été écarté : parmi les répondants, 93%
répondent "Oui" - variance trop faible pour être discriminante. `BD3` (allaitement
en cours), mieux réparti (69%/31%), est conservé seul.

### `wm.sav` - Mère

| Code | Signification |
|------|----------------|
| `HH1`, `HH2`, `LN` | Identifiants (grappe, ménage, ligne de la femme) |
| `CM11` | Nombre total de naissances vivantes de la mère |

---

**Questionnaire de terrain retenu : 13 questions** (hors identifiants techniques
et hors mesures anthropométriques, qui ne servent qu'à construire la cible
pendant l'entraînement, pas à l'usage futur de l'outil) :

`HH6`, `HH7`, `WS1`, `WS11`, `windex5`, `HH48`, `HL4`, `CAGE`, `melevel`, `BD3`,
`CA1`, `CA14`, `CM11`


In [3]:
colonnes_hh = [
    "HH1", "HH2",
    "HH6", "HH7", "WS1", "WS11", "windex5", "HH48",
]

colonnes_ch = [
    "HH1", "HH2", "AN3", "AN5",
    "HL4", "CAGE", "melevel", "BD3", "CA1", "CA14",
    "HAZ2", "WAZ2", "WHZ2", "HAZFLAG", "WAZFLAG", "WHZFLAG", "FLAG",
]

colonnes_wm = [
    "HH1", "HH2", "LN",
    "CM11",
]

hh_selection = hh[colonnes_hh].copy()
ch_selection = ch[colonnes_ch].copy()
wm_selection = wm[colonnes_wm].copy()

print("Dimensions après sélection des variables")
print("-" * 40)
print(f"hh_selection : {hh_selection.shape[0]} lignes, {hh_selection.shape[1]} colonnes")
print(f"ch_selection : {ch_selection.shape[0]} lignes, {ch_selection.shape[1]} colonnes")
print(f"wm_selection : {wm_selection.shape[0]} lignes, {wm_selection.shape[1]} colonnes")


Dimensions après sélection des variables
----------------------------------------
hh_selection : 8404 lignes, 8 colonnes
ch_selection : 5030 lignes, 17 colonnes
wm_selection : 7657 lignes, 4 colonnes


## 3. Décodage des principales variables catégorielles

Rappel pour la lecture des résultats ci-dessous (voir le README pour la liste complète) :

- **HH6** : `1` = Urbain, `2` = Rural
- **HH7** : `1` = Maritime, `2` = Plateaux, `3` = Centrale, `4` = Kara, `5` = Savanes,
  `6` = Lomé Commune, `7` = Golfe Urbain
- **melevel** : `0` = Aucun/Préscolaire, `1` = Primaire, `2` = Secondaire et plus, `9` = Manquant
- **windex5** : `1` = Le plus pauvre → `5` = Le plus riche
- **HL4** : `1` = Masculin, `2` = Féminin
- **BD3 / CA1 / CA14** : `1` = Oui, `2` = Non, `8`/`9` = Ne sait pas / Non réponse


In [4]:
# Aperçu rapide des modalités réellement présentes dans les données
for col in ["HH6", "HH7", "windex5"]:
    print(f"--- {col} ---")
    print(hh_selection[col].value_counts(dropna=False).sort_index())
    print()


--- HH6 ---
HH6
1.0    3340
2.0    5064
Name: count, dtype: int64

--- HH7 ---
HH7
1.0    1200
2.0    1200
3.0    1204
4.0    1200
5.0    1200
6.0    1200
7.0    1200
Name: count, dtype: int64

--- windex5 ---
windex5
0.0     488
1.0    1563
2.0    1482
3.0    1766
4.0    1541
5.0    1564
Name: count, dtype: int64



**Anomalie repérée en vérifiant les sorties ci-dessus** : `windex5` contient un
code `0`, qui n'a **aucun libellé** dans les métadonnées SPSS (seuls `1` à `5`
sont définis : "Le plus pauvre" à "Le plus riche"). C'est un code résiduel,
probablement des ménages non couverts par le calcul de l'indice de richesse -
à traiter comme manquant, pas comme une sixième catégorie valide.


In [5]:
n_avant_windex = hh_selection["windex5"].isna().sum()
hh_selection["windex5"] = hh_selection["windex5"].replace(0, np.nan)
n_apres_windex = hh_selection["windex5"].isna().sum()

print(f"windex5 - NaN avant : {n_avant_windex}, NaN après recodage du 0 : {n_apres_windex}")


windex5 - NaN avant : 0, NaN après recodage du 0 : 488


## 4. Valeurs manquantes : recodage des codes 7/8/9

Les codes `7` (incohérent), `8` (ne sait pas) et `9` (non réponse) sont des codes
de non-réponse, pas des modalités valides. Vérifié dans les métadonnées SPSS :
ces codes ne sont **pas** marqués automatiquement comme manquants par le fichier
`.sav` lui-même - c'est à nous de le faire.

On ne recode ici que les variables catégorielles concernées par ce système de
codes (pas les variables continues comme `CAGE`, `HAZ2`, etc., qui ont leur
propre logique de valeur manquante, déjà en `NaN` dans le fichier d'origine).


In [6]:
codes_non_reponse = [7, 8, 9]
colonnes_a_recoder = ["melevel", "BD3", "CA1", "CA14"]

avant = {c: ch_selection[c].isna().sum() for c in colonnes_a_recoder}

for col in colonnes_a_recoder:
    ch_selection[col] = ch_selection[col].replace(codes_non_reponse, np.nan)

apres = {c: ch_selection[c].isna().sum() for c in colonnes_a_recoder}

print(f"{'Variable':10s} {'NaN avant':>10s} {'NaN après':>10s}")
for c in colonnes_a_recoder:
    print(f"{c:10s} {avant[c]:>10d} {apres[c]:>10d}")


Variable    NaN avant  NaN après
melevel            88         89
BD3              2298       2298
CA1                88         94
CA14               88         93


## 5. Typage des variables catégorielles

Le CSV/pandas ne fait pas spontanément la différence entre une variable nominale
(catégorie, ex. la région) et une variable continue (ex. un z-score) : tout reste
un `float64`. On type explicitement les variables catégorielles pour éviter
qu'un modèle interprète, par exemple, la région `6` comme "plus grande" que la
région `1`. Le passage effectif en variables indicatrices (`OneHotEncoder`) se
fera dans le pipeline de modélisation, pas ici - cette étape ne fait que
documenter/typer les colonnes.


In [7]:
categorielles_hh = ["HH6", "HH7", "WS1", "WS11", "windex5"]
categorielles_ch = ["HL4", "melevel", "BD3", "CA1", "CA14"]

for col in categorielles_hh:
    hh_selection[col] = hh_selection[col].astype("category")

for col in categorielles_ch:
    ch_selection[col] = ch_selection[col].astype("category")

print(ch_selection[categorielles_ch].dtypes)


HL4        category
melevel    category
BD3        category
CA1        category
CA14       category
dtype: object


## 6. Jointure des trois fichiers

On part de `ch_selection` (table pivot, une ligne par enfant) et on vient y
greffer successivement les colonnes du ménage puis de la mère. Le nombre de
lignes ne doit **jamais** bouger par rapport à `ch_selection` : c'est le
contrôle systématique à faire après chaque jointure.


In [8]:
n_avant = len(ch_selection)

df = ch_selection.merge(hh_selection, on=["HH1", "HH2"], how="left")
assert len(df) == n_avant, "La jointure avec hh_selection a modifié le nombre de lignes !"

df = df.merge(
    wm_selection,
    left_on=["HH1", "HH2", "AN5"],
    right_on=["HH1", "HH2", "LN"],
    how="left",
)
assert len(df) == n_avant, "La jointure avec wm_selection a modifié le nombre de lignes !"

print(f"Lignes avant jointure : {n_avant}")
print(f"Lignes après jointure : {len(df)}")
print(f"Colonnes finales : {df.shape[1]}")
df.head()


Lignes avant jointure : 5030
Lignes après jointure : 5030
Colonnes finales : 25


,HH1,HH2,AN3,AN5,HL4,CAGE,melevel,BD3,CA1,CA14,...,WHZFLAG,FLAG,HH6,HH7,WS1,WS11,windex5,HH48,LN,CM11
0,1.0,7.0,5.0,1.0,2.0,33.0,1.0,2.0,2.0,2.0,...,0.0,0.0,1.0,1.0,32.0,12.0,4.0,5.0,1.0,4.0
1,1.0,15.0,2.0,1.0,1.0,41.0,1.0,NaN,2.0,2.0,...,0.0,0.0,1.0,1.0,92.0,22.0,4.0,2.0,1.0,3.0
2,1.0,18.0,6.0,2.0,2.0,12.0,0.0,1.0,2.0,1.0,...,0.0,0.0,1.0,1.0,13.0,23.0,4.0,6.0,2.0,4.0
3,2.0,1.0,4.0,2.0,1.0,22.0,0.0,2.0,2.0,2.0,...,0.0,0.0,1.0,1.0,21.0,23.0,4.0,4.0,2.0,2.0
4,2.0,2.0,2.0,1.0,1.0,3.0,2.0,1.0,1.0,2.0,...,0.0,0.0,1.0,1.0,31.0,22.0,4.0,2.0,1.0,2.0


## 7. Sauvegarde

- Les trois sélections d'origine sont sauvegardées séparément en CSV (transparence,
  lisibles par n'importe qui sans Python).
- Le jeu de données joint et nettoyé est sauvegardé en Parquet (plus rapide à
  recharger, conserve les types `category` et les `NaN` correctement - utile
  pour les itérations rapides pendant le hackathon).

Tout est écrit dans `Stunting_Data/Datasets_convertis/`, laissant `Stunting_Data/Datasets_originaux/` intact
(fichiers `.sav` bruts, jamais modifiés).


In [9]:
hh_selection.to_csv(DOSSIER_CONVERTIS / "hh_selection.csv", index=False)
ch_selection.to_csv(DOSSIER_CONVERTIS / "ch_selection.csv", index=False)
wm_selection.to_csv(DOSSIER_CONVERTIS / "wm_selection.csv", index=False)

df.to_parquet(DOSSIER_CONVERTIS / "donnees_jointes.parquet", index=False)

print(f"Fichiers sauvegardés dans {DOSSIER_CONVERTIS}/ :")
print("- hh_selection.csv, ch_selection.csv, wm_selection.csv (sélections d'origine)")
print("- donnees_jointes.parquet (jeu de données joint, prêt pour la suite)")


Fichiers sauvegardés dans ..\Stunting_Data\Datasets_convertis/ :
- hh_selection.csv, ch_selection.csv, wm_selection.csv (sélections d'origine)
- donnees_jointes.parquet (jeu de données joint, prêt pour la suite)
